# Lecture 6 — Forecast Communication, Monitoring, and Governance
**ECON 417 · Business Forecasting · Southern Illinois University Edwardsville**

Lectures 1–5 built forecasting models.  
Lecture 6 asks: *what happens after you deploy one?*

A model trained on 2015–2019 financial data was deployed in January 2020.  
Two months later, COVID-19 triggered one of the fastest market crashes in history.  
VIX spiked from ~14 to ~80 in six weeks.  
Every model trained on the previous five years was operating in territory it had never seen.

This lecture is about how to detect that failure *before* it causes real damage — and what to do about it.

| Section | Topic |
|---------|-------|
| 1 | From model output to decision communication — the manager dashboard |
| 2 | Monitoring forecast performance after deployment |
| 3 | Data drift and concept drift — the two ways models fail |
| 4 | Champion-challenger evaluation |
| 5 | Overrides, logs, and accountability |
| 6 | From statistical error to business loss |
| 7 | Conceptual introduction to deep learning (LSTM) |
| 8 | Governance checklist |

In [ ]:
%pip install yfinance scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'train':   '#2196F3',
    'val':     '#FF9800',
    'test':    '#4CAF50',
    'ols':     '#E53935',
    'ridge':   '#7B1FA2',
    'lasso':   '#00897B',
    'neutral': '#546E7A',
    'accent':  '#F4A261',
}

GRAPH_DIR = '../Lecture 6/graph'
os.makedirs(GRAPH_DIR, exist_ok=True)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(np.asarray(a), np.asarray(b))))

def mfe(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

In [ ]:
# ── Download financial data (same dataset as L1–L5) ──────────────────────
DATA_SOURCE = 'synthetic (calibrated to real S&P 500 + COVID regime shift)'
raw = None
try:
    import yfinance as yf
    tickers = ['^GSPC', '^VIX', 'MSFT', 'AAPL', 'JPM', 'XOM']
    _raw = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)
    if len(_raw) > 100:
        raw = _raw
        DATA_SOURCE = 'real (yfinance)'
except Exception:
    pass

if raw is not None:
    close = raw['Close']
    sp500 = close['^GSPC']
    vix   = raw['Close']['^VIX'].reindex(sp500.index).ffill()
    msft  = close['MSFT']
    aapl  = close['AAPL']
    jpm   = close['JPM']
    xom   = close['XOM']
else:
    np.random.seed(42)
    dates = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    N = len(dates)
    sp500_ret = np.random.normal(0.0003, 0.010, N)
    covid_start = int(np.searchsorted(dates, pd.Timestamp('2020-02-20')))
    covid_end   = int(np.searchsorted(dates, pd.Timestamp('2020-05-01')))
    crash_len = (covid_end - covid_start) // 2
    sp500_ret[covid_start:covid_start+crash_len] = np.random.normal(-0.025, 0.025, crash_len)
    sp500_ret[covid_start+crash_len:covid_end]   = np.random.normal(+0.015, 0.020, covid_end-covid_start-crash_len)
    sp500 = pd.Series(4000 * np.exp(np.cumsum(sp500_ret)), index=dates)
    vix_base = 18 + np.cumsum(np.random.normal(0, 0.2, N) - 0.3 * sp500_ret * 10)
    vix_base = np.clip(vix_base, 10, 40)
    vix_vals = vix_base.copy()
    vix_vals[covid_start:covid_end] = np.linspace(14, 80, covid_end - covid_start)
    if covid_end + 40 < N:
        vix_vals[covid_end:covid_end+40] = np.linspace(80, 30, 40)
    vix = pd.Series(np.clip(vix_vals, 10, 90), index=dates)
    def make_stock(base, beta, idio):
        r = beta * sp500_ret + np.random.normal(0, idio, N)
        return pd.Series(base * np.exp(np.cumsum(r)), index=dates)
    msft = make_stock(300, 1.2, 0.012)
    aapl = make_stock(150, 1.15, 0.013)
    jpm  = make_stock(80,  1.05, 0.011)
    xom  = make_stock(60,  0.80, 0.014)

df = pd.DataFrame({'SP500': sp500, 'VIX': vix,
                   'MSFT': msft, 'AAPL': aapl, 'JPM': jpm, 'XOM': xom}).dropna()
rets = np.log(df[['SP500','MSFT','AAPL','JPM','XOM']]).diff().dropna()
rets['VIX']     = df['VIX'].reindex(rets.index)
rets['VIX_chg'] = df['VIX'].diff().reindex(rets.index)
rets = rets.dropna()

print(f"Data source : {DATA_SOURCE}")
print(f"Date range  : {rets.index[0].date()} to {rets.index[-1].date()}")
print(f"Observations: {len(rets):,}")

In [ ]:
# ── Governance scenario: model trained 2015-2019, deployed Jan 2020 ───────

X_all = pd.DataFrame({
    'lag_SP500': rets['SP500'].shift(1),
    'lag_MSFT':  rets['MSFT'].shift(1),
    'lag_AAPL':  rets['AAPL'].shift(1),
    'lag_JPM':   rets['JPM'].shift(1),
    'lag_XOM':   rets['XOM'].shift(1),
    'VIX':       rets['VIX'].shift(1),
    'VIX_chg':   rets['VIX_chg'].shift(1),
}).dropna()
y_all    = rets['SP500'].reindex(X_all.index)
FEATURES = list(X_all.columns)

DEPLOY_DATE  = pd.Timestamp('2020-01-02')
train_mask   = X_all.index < DEPLOY_DATE
monitor_mask = X_all.index >= DEPLOY_DATE

X_tr_gov = X_all.loc[train_mask]
y_tr_gov = y_all.loc[train_mask]
X_mon    = X_all.loc[monitor_mask]
y_mon    = y_all.loc[monitor_mask]

# Within training: last 20% = validation
n_val_gov = int(len(X_tr_gov) * 0.20)
X_tr2, y_tr2 = X_tr_gov.iloc[:-n_val_gov], y_tr_gov.iloc[:-n_val_gov]
X_val_gov, y_val_gov = X_tr_gov.iloc[-n_val_gov:], y_tr_gov.iloc[-n_val_gov:]

# Champion: Ridge regression
scaler_gov = StandardScaler().fit(X_tr2)
champion   = Ridge(alpha=50.0)
champion.fit(scaler_gov.transform(X_tr2), y_tr2)

# Challenger: Random forest
challenger = RandomForestRegressor(n_estimators=200, max_depth=5,
                                    max_features='sqrt', random_state=417)
challenger.fit(X_tr2, y_tr2)

# Calibration residuals
val_pred_gov  = champion.predict(scaler_gov.transform(X_val_gov))
val_resid_gov = y_val_gov.values - val_pred_gov
q80_gov       = np.quantile(val_resid_gov, [0.10, 0.90])

# Monitoring predictions
mon_pred_champ = champion.predict(scaler_gov.transform(X_mon))
mon_pred_chall = challenger.predict(X_mon)

baseline_rmse = rmse(y_val_gov, val_pred_gov)
baseline_bias = mfe(y_val_gov, val_pred_gov)
baseline_cov  = np.mean((val_resid_gov >= q80_gov[0]) & (val_resid_gov <= q80_gov[1]))
rmse_alert    = baseline_rmse * 1.5

print(f"Pre-deployment train : {y_tr2.index[0].date()} to {y_tr2.index[-1].date()} ({len(y_tr2):,} obs)")
print(f"Monitoring period    : {y_mon.index[0].date()} to {y_mon.index[-1].date()} ({len(y_mon):,} obs)")
print(f"Baseline RMSE (val)  : {baseline_rmse:.6f}")
print(f"Alert threshold      : {rmse_alert:.6f}")
print(f"80% interval         : [{q80_gov[0]*100:.3f}%, {q80_gov[1]*100:.3f}%]")

---
## Section 1 — From Model Output to Decision Communication

### The manager dashboard

A forecast output is only valuable if the decision-maker can act on it.  
The translation from model output to action requires a clear visual: actual vs. predicted, uncertainty band, and a simple status flag.

A good manager dashboard answers three questions at a glance:
1. **What is the expected path?** (point forecast + recent actuals)
2. **How uncertain is it?** (prediction interval)
3. **Is anything going wrong?** (rolling RMSE traffic-light indicator)

In [ ]:
# ── Example 6.1 — Manager dashboard ──────────────────────────────────────

# Dashboard window: show Jan–May 2020 to capture the COVID crash story
try:
    dash_start = pd.Timestamp('2020-01-02')
    dash_end   = pd.Timestamp('2020-06-30')
    dash_mask  = (y_mon.index >= dash_start) & (y_mon.index <= dash_end)
    if dash_mask.sum() < 20:
        dash_mask = pd.Series([True] * min(120, len(y_mon)) + [False] * (len(y_mon) - min(120, len(y_mon))),
                               index=y_mon.index).values.astype(bool)
except Exception:
    dash_mask = np.ones(min(120, len(y_mon)), dtype=bool)

y_dash    = y_mon[dash_mask]
pred_dash = mon_pred_champ[dash_mask]
lo_dash   = pred_dash + q80_gov[0]
hi_dash   = pred_dash + q80_gov[1]
err_dash  = y_dash.values - pred_dash
roll_rmse_dash = pd.Series(err_dash**2).rolling(10).mean().apply(np.sqrt).values

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.fill_between(y_dash.index, lo_dash, hi_dash,
                 color=COLORS['train'], alpha=0.25, label='80% forecast interval')
ax.plot(y_dash.index, y_dash.values, color=COLORS['neutral'], lw=1.3,
        label='Actual S&P 500 return', alpha=0.9)
ax.plot(y_dash.index, pred_dash, color=COLORS['train'], lw=2.0,
        label='Champion model forecast', zorder=4)
ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
flag = np.abs(err_dash) > np.abs(q80_gov).max() * 1.5
ax.scatter(y_dash.index[flag], y_dash.values[flag],
           color=COLORS['ols'], s=30, zorder=5, label='Large miss (risk flag)', marker='x')
ax.set_ylabel('S&P 500 Log Return', fontsize=11)
ax.set_title('Example 6.1 — Manager Dashboard: S&P 500 Forecast Communication\n'
             'Actual vs forecast + 80% interval + risk flags — the three dashboard questions at a glance',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')

ax = axes[1]
ax.plot(y_dash.index, roll_rmse_dash, color=COLORS['val'], lw=2, label='10-day rolling RMSE')
ax.axhline(baseline_rmse, color=COLORS['test'], lw=1.5, ls='--',
           label=f'Baseline RMSE = {baseline_rmse:.5f}')
ax.axhline(rmse_alert, color=COLORS['ols'], lw=1.5, ls=':',
           label=f'Alert threshold = {rmse_alert:.5f}')
ax.fill_between(y_dash.index, roll_rmse_dash, rmse_alert,
                 where=(roll_rmse_dash > rmse_alert), color=COLORS['ols'], alpha=0.25, label='Alert zone')
ax.set_ylabel('Rolling RMSE', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_1_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_6_1 saved.")

---
## Section 2 — Monitoring Forecast Performance After Deployment

### Why evaluation doesn't stop at the test set

The environment changes after the model is built. A test-set evaluation is a snapshot in time.  
A stable forecasting process continuously tracks three metrics:

1. **Rolling RMSE** — is statistical accuracy deteriorating?
2. **Rolling bias** — is the model systematically over- or under-forecasting?
3. **Rolling coverage** — is the 80% interval still containing ~80% of outcomes?

Thresholds are set *before* deployment so that analysts cannot rationalize bad performance after the fact.

In [ ]:
# ── Example 6.2 — Rolling RMSE, bias, and coverage ───────────────────────

WINDOW = 60
mon_errors = y_mon.values - mon_pred_champ
lo_mon     = mon_pred_champ + q80_gov[0]
hi_mon     = mon_pred_champ + q80_gov[1]
inside_80  = (y_mon.values >= lo_mon) & (y_mon.values <= hi_mon)

roll_rmse     = pd.Series(mon_errors**2).rolling(WINDOW).mean().apply(np.sqrt).values
roll_bias     = pd.Series(mon_errors).rolling(WINDOW).mean().values
roll_coverage = pd.Series(inside_80.astype(float)).rolling(WINDOW).mean().values

cov_alert = 0.65

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Example 6.2 — Rolling Monitoring Metrics After Deployment\n'
             'Model trained 2015–2019, deployed Jan 2020 — watch for the COVID deterioration',
             fontsize=13, fontweight='bold')

# RMSE
ax = axes[0]
ax.plot(y_mon.index, roll_rmse, color=COLORS['train'], lw=2, label=f'{WINDOW}-day rolling RMSE')
ax.axhline(baseline_rmse, color='black', lw=1.5, ls='--', alpha=0.7, label=f'Baseline RMSE = {baseline_rmse:.5f}')
ax.axhline(rmse_alert,    color=COLORS['ols'], lw=1.5, ls=':', label=f'Alert = {rmse_alert:.5f}')
ax.fill_between(y_mon.index, roll_rmse, rmse_alert,
                 where=(roll_rmse > rmse_alert), color=COLORS['ols'], alpha=0.18)
ax.set_ylabel('Rolling RMSE', fontsize=10)
ax.set_title('Statistical Accuracy Monitor', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

# Bias
ax = axes[1]
ax.plot(y_mon.index, roll_bias, color=COLORS['ridge'], lw=2, label=f'{WINDOW}-day rolling bias')
ax.axhline(0,                   color='black', lw=1.0, ls='--', alpha=0.5, label='Zero bias')
ax.axhline( baseline_rmse*0.5,  color=COLORS['ols'], lw=1.2, ls=':', label='Bias alert')
ax.axhline(-baseline_rmse*0.5,  color=COLORS['ols'], lw=1.2, ls=':')
ax.fill_between(y_mon.index, roll_bias, 0,
                 where=(np.abs(roll_bias) > baseline_rmse*0.5),
                 color=COLORS['ols'], alpha=0.15)
ax.set_ylabel('Mean Forecast Error', fontsize=10)
ax.set_title('Bias Monitor (0 = ideal)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

# Coverage
ax = axes[2]
ax.plot(y_mon.index, roll_coverage, color=COLORS['lasso'], lw=2, label=f'{WINDOW}-day coverage')
ax.axhline(0.80,     color='navy',       lw=1.5, ls='--', alpha=0.5, label='Target 80%')
ax.axhline(cov_alert, color=COLORS['ols'], lw=1.5, ls=':', label=f'Alert < {cov_alert:.0%}')
ax.fill_between(y_mon.index, roll_coverage, cov_alert,
                 where=(roll_coverage < cov_alert), color=COLORS['ols'], alpha=0.18, label='Under-coverage')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Coverage Rate', fontsize=10)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('Interval Coverage Monitor', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_2_monitoring_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_6_2 saved.")

---
## Section 3 — Drift and Forecast Failure Signals

### Two ways a deployed model fails

**Data drift**: the distribution of input features shifts away from what the model was trained on.  
Example: VIX averaged 14 during training. Post-COVID it averaged 25–35.  
The model operates on inputs it never encountered during training.

**Concept drift**: the relationship between inputs and the target changes.  
Example: a 1-point VIX rise predicted a −0.5% return during 2015–2019.  
During the COVID crash, the VIX–return relationship broke down (VIX was already at 70; one more point was noise).

| | Data drift | Concept drift |
|--|-----|------|
| What changes | Distribution of X | Relationship X → y |
| COVID example | VIX shifts from 14 to 70 | VIX-return sensitivity breaks down |
| Detection | Compare input distributions | Watch coefficient stability and residuals |
| Response | Retrain with recent data | Reconsider model structure |

In [ ]:
# ── Example 6.3 — Data drift (VIX distribution) + concept drift ──────────

pre_mask  = X_all.index < '2020-02-01'
cov_mask  = (X_all.index >= '2020-02-01') & (X_all.index < '2020-07-01')
post_mask = (X_all.index >= '2020-07-01') & (X_all.index < '2022-01-01')

vix_pre   = X_all.loc[pre_mask,  'VIX']
vix_cov   = X_all.loc[cov_mask,  'VIX']
vix_post  = X_all.loc[post_mask, 'VIX']

# Rolling VIX_chg coefficient for concept drift
WINDOW_D = 120
roll_coef, roll_dates = [], []
for end in range(WINDOW_D, len(X_all)):
    Xw = X_all.iloc[end-WINDOW_D:end][['VIX_chg']]
    yw = y_all.iloc[end-WINDOW_D:end]
    m  = LinearRegression().fit(Xw, yw)
    roll_coef.append(m.coef_[0])
    roll_dates.append(X_all.index[end])
roll_coef_s = pd.Series(roll_coef, index=roll_dates)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Example 6.3 — Data Drift and Concept Drift: The COVID Regime Shift',
             fontsize=13, fontweight='bold')

# VIX distribution
ax = axes[0, 0]
ax.hist(vix_pre,  bins=30, density=True, alpha=0.7, color=COLORS['test'],
        label=f'Pre-COVID (avg VIX={vix_pre.mean():.1f})',    edgecolor='white')
ax.hist(vix_cov,  bins=20, density=True, alpha=0.7, color=COLORS['ols'],
        label=f'COVID peak (avg VIX={vix_cov.mean():.1f})',  edgecolor='white')
ax.hist(vix_post, bins=30, density=True, alpha=0.7, color=COLORS['val'],
        label=f'Post-COVID (avg VIX={vix_post.mean():.1f})', edgecolor='white')
ax.set_xlabel('VIX Level', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Data Drift: VIX Distribution Shifts\nModel trained on far-left tail; crisis pushes to right', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

# VIX time series
ax = axes[0, 1]
ax.plot(X_all.index, X_all['VIX'], color=COLORS['neutral'], lw=0.8, alpha=0.85)
ax.axhline(vix_pre.mean(), color=COLORS['test'], lw=2, ls='--',
           label=f'Training avg VIX = {vix_pre.mean():.1f}')
ax.axhline(28, color=COLORS['ols'], lw=1.5, ls=':', label='Data drift alert (VIX > 28)')
try:
    ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-07-01'),
               alpha=0.18, color=COLORS['ols'], label='COVID crisis window')
except Exception:
    pass
ax.set_ylabel('VIX Level', fontsize=11)
ax.set_title('VIX Over Time\nCrisis = model operating far outside training distribution', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

# Rolling VIX_chg coefficient
ax = axes[1, 0]
ax.plot(roll_coef_s.index, roll_coef_s.values, color=COLORS['ridge'], lw=1.8, alpha=0.9)
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.4)
try:
    ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-07-01'),
               alpha=0.18, color=COLORS['ols'])
except Exception:
    pass
ax.set_ylabel('OLS Coefficient on VIX_chg', fontsize=11)
ax.set_title('Concept Drift: VIX-Return Relationship\n'
             'Coefficient swings during crisis = relationship broke down', fontsize=11, fontweight='bold')

# Monitoring RMSE
ax = axes[1, 1]
ax.plot(y_mon.index, roll_rmse, color=COLORS['train'], lw=2, label=f'{WINDOW}-day RMSE')
ax.axhline(baseline_rmse, color=COLORS['test'], lw=1.5, ls='--', label='Pre-deployment RMSE')
ax.axhline(rmse_alert,    color=COLORS['ols'],  lw=1.5, ls=':', label='Alert threshold')
ax.fill_between(y_mon.index, roll_rmse, rmse_alert,
                 where=(roll_rmse > rmse_alert), color=COLORS['ols'], alpha=0.20, label='Failure signal')
ax.set_ylabel('Rolling RMSE', fontsize=11)
ax.set_title('Monitoring Catches the Drift\nRMSE rises when regime shifts', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)

for ax in axes.flat:
    ax.set_xlabel('Date', fontsize=10)

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_3_drift_signal.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_6_3 saved.")

---
## Section 4 — Champion-Challenger Evaluation

### Never replace a model based on one lucky week

The champion is the current production model (Ridge, trained pre-deployment).  
The challenger is a competing model tracked in parallel (Random Forest).  

Governance rule: require the challenger to outperform **consistently** over a meaningful window  
before promotion — not just on one good stretch.  
Frequent model switches create operational noise and erode trust from traders and risk committees.

In [ ]:
# ── Example 6.4 — Champion vs. challenger rolling RMSE ────────────────────

EVAL_WIN = 60
champ_err = y_mon.values - mon_pred_champ
chall_err = y_mon.values - mon_pred_chall

roll_rmse_champ = pd.Series(champ_err**2).rolling(EVAL_WIN).mean().apply(np.sqrt).values
roll_rmse_chall = pd.Series(chall_err**2).rolling(EVAL_WIN).mean().apply(np.sqrt).values

challenger_winning = roll_rmse_chall < roll_rmse_champ

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Example 6.4 — Champion vs. Challenger: Rolling RMSE Comparison\n'
             'Champion = Ridge regression | Challenger = Random forest',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(y_mon.index, roll_rmse_champ, color=COLORS['train'],   lw=2.2, label='Champion (Ridge)')
ax.plot(y_mon.index, roll_rmse_chall, color=COLORS['ols'],     lw=2.2, label='Challenger (Random forest)', ls='--')
ax.fill_between(y_mon.index, roll_rmse_champ, roll_rmse_chall,
                 where=challenger_winning, color=COLORS['ols'],   alpha=0.15, label='Challenger leads')
ax.fill_between(y_mon.index, roll_rmse_champ, roll_rmse_chall,
                 where=~challenger_winning, color=COLORS['train'], alpha=0.10, label='Champion leads')
ax.set_ylabel(f'{EVAL_WIN}-Day Rolling RMSE', fontsize=11)
ax.set_title('Rolling RMSE — Champion vs. Challenger', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
rmse_diff = roll_rmse_champ - roll_rmse_chall
ax.plot(y_mon.index, rmse_diff, color=COLORS['neutral'], lw=1.5)
ax.fill_between(y_mon.index, rmse_diff, 0, where=(rmse_diff > 0),
                 color=COLORS['ols'],   alpha=0.28, label='Challenger better (consider promotion)')
ax.fill_between(y_mon.index, rmse_diff, 0, where=(rmse_diff <= 0),
                 color=COLORS['train'], alpha=0.18, label='Champion better (keep)')
ax.axhline(0, color='black', lw=1.0, ls='--')
ax.set_ylabel('Champion − Challenger RMSE', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('RMSE Differential  (positive = challenger winning)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_4_champion_challenger.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Challenger leads in {challenger_winning.mean()*100:.0f}% of rolling windows.")
print("Governance rule: require 60+ consecutive days of outperformance before promotion.")

---
## Section 5 — Overrides, Logs, and Accountability

### Human judgment should be governed, not informal

An analyst may override the model before a known Fed meeting, earnings announcement, or geopolitical event.  
Without a log, we only remember the overrides that worked — a classic hindsight bias.  
With a log, we can objectively evaluate whether human interventions improve forecast quality over time.

In [ ]:
# ── Example 6.5 — Override log ────────────────────────────────────────────

np.random.seed(7)
override_dates = y_mon.index[::max(1, int(len(y_mon)/6))][:5]

override_log = []
for i, date in enumerate(override_dates):
    idx       = y_mon.index.get_loc(date)
    model_fc  = mon_pred_champ[idx]
    actual    = y_mon.iloc[idx]
    if i % 2 == 0:
        override = model_fc + 0.5*(actual - model_fc) + np.random.normal(0, 0.001)
        reason   = 'Fed meeting: cautious upward adjustment' if actual > model_fc else 'Earnings risk: downward adjustment'
    else:
        override = model_fc - 0.4*(actual - model_fc) + np.random.normal(0, 0.001)
        reason   = 'Macro optimism: raised forecast'
    override_log.append({
        'Date':             date.strftime('%Y-%m-%d'),
        'Model forecast':   f"{model_fc*100:+.3f}%",
        'Override':         f"{override*100:+.3f}%",
        'Actual':           f"{actual*100:+.3f}%",
        'Override |error|': f"{abs(actual-override)*100:.4f}%",
        'Model |error|':    f"{abs(actual-model_fc)*100:.4f}%",
        'Override better?': '✓ YES' if abs(actual-override) < abs(actual-model_fc) else '✗ NO',
        'Reason':           reason,
    })

log_df = pd.DataFrame(override_log)
print("=== Override Log ===")
print(log_df[['Date','Model forecast','Override','Actual','Override better?','Reason']].to_string(index=False))

# Plot
W_over = min(100, len(y_mon))

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_mon.index[:W_over], y_mon.values[:W_over],
        color=COLORS['neutral'], lw=1.3, alpha=0.9, label='Actual return')
ax.plot(y_mon.index[:W_over], mon_pred_champ[:W_over],
        color=COLORS['train'], lw=1.8, label='Champion model forecast', zorder=4)

for row in override_log:
    try:
        ov_date = pd.Timestamp(row['Date'])
        ov_val  = float(row['Override'].replace('%','')) / 100
        col = COLORS['test'] if row['Override better?'].startswith('✓') else COLORS['ols']
        marker = '^' if ov_val > float(row['Model forecast'].replace('%',''))/100 else 'v'
        if ov_date in y_mon.index[:W_over]:
            ax.axvline(ov_date, color=col, lw=1.3, ls=':', alpha=0.7)
            ax.scatter([ov_date], [ov_val], color=col, s=90, zorder=6, marker=marker)
    except Exception:
        pass

ax.axhline(0, color='black', lw=0.7, ls='--', alpha=0.4)
legend_elements = [
    plt.Line2D([0],[0], color=COLORS['neutral'], lw=1.5, label='Actual return'),
    plt.Line2D([0],[0], color=COLORS['train'],   lw=2.0, label='Champion forecast'),
    mpatches.Patch(color=COLORS['test'], alpha=0.7, label='Override improved accuracy ✓'),
    mpatches.Patch(color=COLORS['ols'],  alpha=0.7, label='Override hurt accuracy ✗'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('S&P 500 Log Return', fontsize=11)
ax.set_title('Example 6.5 — Override Log: Are Analyst Interventions Improving Forecasts?\n'
             'Without the log, we only remember the overrides that worked',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_5_override_log.png', dpi=150, bbox_inches='tight')
plt.show()
n_better = sum(1 for r in override_log if r['Override better?'].startswith('✓'))
print(f"\n{n_better}/{len(override_log)} overrides improved accuracy — log enables this audit.")

---
## Section 6 — From Statistical Error to Business Loss

### Two monitoring lenses: RMSE and dollar impact

Statistical metrics measure error in return units. Business loss measures dollar impact.  
They can diverge: RMSE might be stable while business loss grows — if the cost structure of errors changes.

Using the asymmetric loss framework from Lecture 5:
$$\overline{\text{Loss}} = \frac{1}{T}\sum_{t=1}^T \left[ c_{\text{under}} \cdot \max(e_t, 0) + c_{\text{over}} \cdot \max(-e_t, 0) \right]$$

Reporting *"our model costs the fund an estimated $X/day in excess losses"* is often more compelling to decision-makers than reporting RMSE.

In [ ]:
# ── Example 6.6 — Business loss alongside RMSE ────────────────────────────

C_UNDER, C_OVER = 1.0, 3.0
PORTFOLIO = 1_000_000   # $1M reference portfolio

def biz_loss(errors, c_u, c_o, port):
    return np.where(errors > 0, c_u * np.abs(errors), c_o * np.abs(errors)) * port

daily_loss_champ = biz_loss(champ_err, C_UNDER, C_OVER, PORTFOLIO)
daily_loss_chall = biz_loss(chall_err, C_UNDER, C_OVER, PORTFOLIO)

roll_loss_champ = pd.Series(daily_loss_champ).rolling(EVAL_WIN).mean().values
roll_loss_chall = pd.Series(daily_loss_chall).rolling(EVAL_WIN).mean().values

baseline_loss = np.mean(biz_loss(val_resid_gov, C_UNDER, C_OVER, PORTFOLIO))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Example 6.6 — Statistical Error vs. Business Loss: Two Monitoring Lenses\n'
             f'Portfolio = ${PORTFOLIO:,} | c_over = {C_OVER}× c_under',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(y_mon.index, roll_rmse, color=COLORS['train'], lw=2, label='Champion rolling RMSE')
ax.axhline(baseline_rmse, color='black', lw=1.5, ls='--', alpha=0.6, label='Baseline')
ax.axhline(rmse_alert,    color=COLORS['ols'], lw=1.5, ls=':', label='Alert threshold')
ax.fill_between(y_mon.index, roll_rmse, rmse_alert,
                 where=(roll_rmse > rmse_alert), color=COLORS['ols'], alpha=0.18)
ax.set_ylabel('Rolling RMSE', fontsize=11)
ax.set_title('Statistical Monitor: Rolling RMSE', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
ax.plot(y_mon.index, roll_loss_champ, color=COLORS['ridge'],  lw=2,       label='Champion avg daily loss')
ax.plot(y_mon.index, roll_loss_chall, color=COLORS['accent'],  lw=2, ls='--', label='Challenger avg daily loss')
ax.axhline(baseline_loss, color='black', lw=1.5, ls='--', alpha=0.6,
           label=f'Baseline daily cost = ${baseline_loss:,.0f}')
ax.set_ylabel(f'Avg Daily Business Loss ($)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title(f'Business Loss Monitor ($-impact per day)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_6_business_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Pre-deployment avg daily cost: ${baseline_loss:,.2f}")
print(f"Full monitoring champion cost: ${np.nanmean(roll_loss_champ):,.2f}/day")

---
## Section 7 — Conceptual Introduction to Deep Learning (LSTM)

### When would you move beyond this course's methods?

An **LSTM (Long Short-Term Memory)** is a type of recurrent neural network that learns patterns from long sequences.  
Instead of an analyst specifying AR(p) orders or interaction terms, an LSTM learns the relevant structure from data.

**The key insight is not that LSTM is better or worse:**  
it is that the *same validation discipline always applies*.  
Train/validation/test split, out-of-sample RMSE, uncertainty quantification, and monitoring after deployment  
are all still required, regardless of model complexity.

| Feature | Methods in L1–L6 | LSTM |
|---------|-----------------|------|
| Interpretability | High (coefficients, rules) | Low (millions of weights) |
| Min data needed | ~100–500 obs | >5,000+ obs |
| Validation rule | Train/val/test | **Identical — same rules** |
| Forecast intervals | Empirical quantiles | Requires additional techniques |
| Regulatory risk | Low | High |

**Practical decision rule:**
1. Start with the simplest model that beats baselines out of sample.
2. Add complexity only when validation clearly justifies it.
3. Use LSTM when: dataset is large, simpler models leave unexplained structure, AND the context tolerates lower interpretability.

In [ ]:
# ── Example 6.7 — LSTM context: where it fits ────────────────────────────

models_info = [
    ('AR(p)',        'L1',  1,   5,  1.0, COLORS['test'],    'Risk mgmt'),
    ('Ridge',        'L3',  2,   4,  1.5, COLORS['ridge'],   'Factor models'),
    ('Rand. Forest', 'L4',  3.5, 2.5, 3.5, COLORS['train'], 'Credit scoring'),
    ('LSTM',         '—',   5,   1,  5.0, COLORS['ols'],    'HFT / NLP'),
]

fig, ax = plt.subplots(figsize=(10, 5.5))
for name, lect, complexity, interp, data_sz, col, use_case in models_info:
    ax.scatter(complexity, interp, s=data_sz*80, color=col, alpha=0.85, zorder=5)
    ax.annotate(f'{name}\n({lect})', (complexity, interp),
                textcoords='offset points', xytext=(10, 4),
                fontsize=11, fontweight='bold', color=col)
    ax.annotate(f'Use: {use_case}', (complexity, interp),
                textcoords='offset points', xytext=(10, -10),
                fontsize=9, color='gray')

ax.set_xlabel('Model Complexity  →', fontsize=12)
ax.set_ylabel('←  Interpretability', fontsize=12)
ax.set_xlim(0, 6.5)
ax.set_ylim(0, 6.5)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title('Example 6.7 — Where LSTM Fits in the Complexity Spectrum\n'
             'Bubble size ∝ minimum data requirement | This course = left-to-center',
             fontsize=12, fontweight='bold')

# Course coverage arrow
ax.annotate('', xy=(4.0, 5.8), xytext=(0.8, 5.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(2.4, 6.0, 'This course', fontsize=10, color='gray', ha='center')

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/fig_6_7_lstm_context.png', dpi=150, bbox_inches='tight')
plt.show()
print("fig_6_7 saved.")

# Print the comparison table
comp = pd.DataFrame({
    'Model':           ['AR(p)', 'Ridge', 'Random Forest', 'LSTM'],
    'Lectures':        ['L1',    'L3',    'L4',            '(next step)'],
    'Min data (days)': ['~50',   '~100',  '~500',          '>5000'],
    'Interpretable':   ['Yes',   'Yes',   'Partial',       'No'],
    'Regulatory risk': ['Low',   'Low',   'Medium',        'High'],
})
print("\nModel complexity comparison:")
print(comp.to_string(index=False))

---
## Section 8 — Governance Checklist

### Five questions every deployed model must answer

Before declaring a model ready for real decisions, a forecaster should be able to answer **all five** of the following:

1. **What data were used?** Source, time range, preprocessing steps, data quality issues.
2. **What validation rule was followed?** Chronological split, proportions, no look-ahead.
3. **What uncertainty measure was reported?** Interval width, coverage rate, residual distribution check.
4. **What monitoring threshold was chosen?** Rolling RMSE or bias level that triggers a review.
5. **What would trigger re-estimation?** Specific drift signal, coverage drop, regime event, or calendar date.

In [ ]:
# ── Example 6.8 — Governance checklist printout ────────────────────────────

checklist = {
    '1. Data':       (f"yfinance: S&P 500, VIX, MSFT, AAPL, JPM, XOM (2015–2019). "
                      f"Daily log returns + VIX level. All predictors lagged 1 day. "
                      f"VIX forward-filled on holidays. {len(y_tr2):,} training observations."),
    '2. Validation': (f"Chronological split within 2015–2019: 80% train / 20% val. "
                      f"No look-ahead: features lagged 1 day. "
                      f"λ=50 chosen on validation RMSE. Test set (2020+) not used during development."),
    '3. Uncertainty':(f"Empirical 80% interval from validation OOS residuals: "
                      f"[{q80_gov[0]*100:.3f}%, {q80_gov[1]*100:.3f}%]. "
                      f"Q-Q plot inspected for fat tails (empirical quantiles used, not normal formula)."),
    '4. Monitoring': (f"60-day rolling RMSE alert at {rmse_alert:.6f} (1.5× baseline {baseline_rmse:.6f}). "
                      f"Bias alert if |mean error| > {baseline_rmse*0.5:.6f}. "
                      f"Coverage alert if 80% interval < 65% over 60 days."),
    '5. Re-estimation trigger': (f"Any of: RMSE > alert threshold 10+ consecutive days; "
                      f"VIX mean > 28 for 1 week (data drift); "
                      f"Coverage < 65% for 2 weeks; "
                      f"Confirmed macro regime change; "
                      f"Scheduled semi-annual review (June / December)."),
}

print("╔══════════════════════════════════════════════════════════════════════════╗")
print("║  GOVERNANCE CHECKLIST  |  S&P 500 Daily Return Forecasting Model        ║")
print("║  Champion: Ridge Regression (λ=50)  |  Deployed: Jan 2020               ║")
print("╠══════════════════════════════════════════════════════════════════════════╣")
for key, value in checklist.items():
    print(f"\n  {key}:")
    for line in textwrap.wrap(value, width=74):
        print(f"    {line}")
print()
print("╚══════════════════════════════════════════════════════════════════════════╝")
print()
print("STATUS: ✓ All 5 governance questions answered.")
print("        Model is deployment-ready with a defined maintenance plan.")

---
## Forecast Memo Template

---

**FORECAST MEMO**  
*S&P 500 Daily Return | Champion Ridge Model | Prepared: [Date]*

**Forecast:** Point = `+0.25%` | 80% interval = `[−1.1%, +1.6%]`

**Model status:** ✅ Within normal operating range  
Rolling RMSE (60-day) is within 1.5× baseline | 80% coverage = 78% (close to 80% target)

**Risk flags:**  
- VIX currently 22 — within training distribution.  
- No scheduled Fed announcements in next 3 days.  
- No active analyst override.

**Recommendation:**  
Maintain current equity exposure.  
Monitor daily: if VIX rises above 28 for 3 consecutive days, trigger model review protocol.

---
## Interview Preparation

**"How do you evaluate forecast quality?"**  
> "I evaluate on three dimensions: statistical accuracy (out-of-sample RMSE and MAE), uncertainty calibration (whether prediction intervals cover outcomes at the stated rate), and business impact (forecast errors translated to dollar cost using an asymmetric loss function).  
> Evaluation continues *after* deployment through rolling monitoring — a one-time test-set score is not sufficient."

**"Why didn't you use a more complex model?"**  
> "Complexity requires justification, not just availability.  
> Simpler models are easier to explain, audit, and maintain — which matters enormously in regulated financial settings.  
> Ridge regression produced acceptable out-of-sample performance for this daily return task,  
> and the governance cost of a black-box model was not justified by the accuracy gain.  
> If we had much larger data and clear unexplained structure, adding a random forest or LSTM would be revisited."

**"What would make you change your approach?"**  
> "I would revisit the model if: rolling RMSE rises above 1.5× baseline for 10+ consecutive days; bias drifts persistently in one direction; or coverage drops below 65%.  
> I would also reconsider after a confirmed regime shift — such as the COVID crash, a major Fed policy pivot, or a structural market microstructure change.  
> The first diagnostic question would be: is this data drift (inputs shifted) or concept drift (X→y relationship changed)? Because those two problems require different responses."